# Chatbot with Similarity Score

This notebook uses the fundamental code of the rubin_rag chatbot to perform similarity searches and return similarity scores along with the RAG LLM response. It is missing the chat history functionality of the full chatbot, but the user can receive a single answer to a single query. This is useful for viewing the quality of context retrieved by the RAG and the relation of the answer provided by the LLM to that retrieved context.

In [ ]:
import os
from dotenv import load_dotenv
import warnings
import time
import logging

import weaviate
from weaviate.classes.query import Filter

from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_core.prompts import MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from weaviate.classes.init import Auth


from langchain_core.documents.base import Document
from langchain_core.runnables import Runnable
from langchain_core.runnables import RunnableLambda

from langchain_weaviate.vectorstores import WeaviateVectorStore
from typing import List, Callable

In [ ]:
# Suppress all warnings
warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
def configure_custom_retriever(client) -> Callable[[dict], List[Document]]:
    vectorstore = WeaviateVectorStore(
        client=client,
        # index_name="LangChain_9787ec4b92d3438a8de3ff04ead7ead6",
        index_name="Ingestion_20250610",
        text_key="page_content",
        embedding=OpenAIEmbeddings(model="text-embedding-3-small",
                                   dimensions=1536
        ),
        attributes=["source", "source_key"],
    )

    sources = ["github", "jira", "lsst_bib", "webpage", "discourse"]
    filters = Filter.by_property("source_key").contains_any(sources)

    def retrieve(inputs: dict) -> List[Document]:
        query = inputs["input"]

        results = vectorstore.similarity_search_with_score(
            query=query,
            k=6,
            filters=filters,
        )
        docs = []
        for doc, score in results:
            doc.metadata["similarity_score"] = score
            docs.append(doc)
        return docs

    return retrieve


def create_qa_chain(
    input_retriever: Callable[[str], List[Document]],
) -> Runnable:
    """Create a QA chain for the chatbot using a custom retriever."""

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

    system_template = """You are Rubin AI Assistant, a helpful assistant at
    Vera C Rubin Observatory.
    Do your best to answer the questions in as much detail as possible.
    Do not attempt to provide an answer if you do not know the answer.
    In your response, do not recommend reading elsewhere.
    Use the following pieces of context to answer the user's
    question at the end.
    ----------------
    {context}
    ----------------"""

    qa_prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template(system_template),
            MessagesPlaceholder("chat_history"),
            HumanMessagePromptTemplate.from_template("Question:```{input}```"),
        ]
    )

    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
    
    retriever_runnable = RunnableLambda(input_retriever)
    return create_retrieval_chain(retriever_runnable, question_answer_chain)

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host="weaviate-headless.rubin-rag.svc.cluster.local",
        http_port=8080,
        http_secure=False,
        grpc_host="weaviate-grpc.rubin-rag.svc.cluster.local",
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(os.getenv("WEAVIATE_API_KEY")),
        headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")},
        skip_init_checks=True,
    )
    
    start = time.time()
    retriever = configure_custom_retriever(client)
    elapsed = time.time() - start
    print(f"Configure retriever took {elapsed:.2f}s")
    
    start = time.time()
    qa_chain = create_qa_chain(retriever)
    elapsed = time.time() - start
    print(f"Create QA chain took {elapsed:.2f}s")

    query = "instantiate lsst.daf.butler.Butler in Python tutorial?"

    start = time.time()
    result = qa_chain.invoke({
        "input": query,
        "chat_history": [],
    })
    elapsed = time.time() - start
    print(f"Invoke took {elapsed:.2f}s")

    print("Input:\n", result["input"])
    print("\nAnswer:\n", result["answer"])
    
    print("\nContext Documents:")
    for i, doc in enumerate(result["context"], 1):
        print(f"\n--- Document {i} ---")
        print("Source:", doc.metadata.get("source"))
        print("Repo:", doc.metadata.get("repo"))
        print("Similarity Score:", doc.metadata.get("similarity_score"))
        print("Content:\n", doc.page_content)


except Exception as e:
    print(f"An error occurred: {e}")
finally:
    client.close()